In [1]:
!pip install -q transformers datasets peft accelerate bitsandbytes trl rouge_score nltk evaluate sacrebleu
import os
import gc
import torch
import numpy as np
import pandas as pd
import nltk
# Replace the two evaluate.load() lines with this:
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 9.1 MB/s eta 0:00:00


In [2]:
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

print("PyTorch version :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU             :", torch.cuda.get_device_name(0))
    print("VRAM            :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")


PyTorch version : 2.10.0+cu128
CUDA available  : True
GPU             : Tesla T4
VRAM            : 15.6 GB


In [3]:
"""
HARDWARE FEASIBILITY JUSTIFICATION
===================================
Full Fine-Tuning (FFT) of a 7B-parameter model requires approximately 120 GB of
GPU memory for optimizer states, activations, and gradients when using Adam in
fp32. The Google Colab free tier provides a single NVIDIA T4 with 15 GB VRAM,
which is insufficient for FFT of any model larger than ~1B parameters.

TinyLlama-1.1B in fp32 would still require ~16 GB for FFT (4 bytes × 1.1B params
for weights + similar for optimizer states), marginally exceeding the T4's limit.

Therefore, Full Fine-Tuning is infeasible on this hardware. The primary
adaptation method used is QLoRA (4-bit quantisation + LoRA adapters), which
reduces trainable parameters to ~0.5% of the total and fits comfortably within
15 GB VRAM. Results are compared against the untuned base model (Baseline).
"""
print("Hardware assessment complete — proceeding with QLoRA.")

Hardware assessment complete — proceeding with QLoRA.


In [4]:
dataset = load_dataset("sugiv/synthetic-text-transformation-dataset", split="train")
print("Raw dataset size:", len(dataset))
print("Columns:", dataset.column_names)

# --- Filter definitions ---
FINANCE_KEYWORDS = [
    "revenue", "profit", "cost", "margin", "forecast",
    "financial", "budget", "pricing", "churn",
    "retention", "growth", "investment", "roi",
    "risk", "liquidity", "cash flow", "variance",
    "performance", "kpi", "analytics", "model",
]

BAD_MARKETING_WORDS = [
    "game-changing", "revolutionary", "cutting-edge",
    "unprecedented opportunity", "unlock massive growth",
    "disruptive innovation",
]

def finance_filter(ex):
    return any(kw in ex["input_text"].lower() for kw in FINANCE_KEYWORDS)

def sentence_length_filter(ex):
    n = len(ex["input_text"].split())
    return 10 <= n <= 60

def length_ratio_filter(ex):
    in_len = len(ex["input_text"].split())
    out_len = len(ex["output_text"].split())
    return in_len > 0 and 0.9 <= (out_len / in_len) <= 1.6

def marketing_filter(ex):
    return not any(w in ex["output_text"].lower() for w in BAD_MARKETING_WORDS)

def tone_complexity_filter(ex):
    return (
        ex.get("Tone") == "Professional"
        and ex.get("Complexity Level") in ["Advanced", "Intermediate"]
    )

# Apply all filters in sequence BEFORE selecting a fixed size
dataset = dataset.filter(finance_filter)
dataset = dataset.filter(sentence_length_filter)
dataset = dataset.filter(length_ratio_filter)
dataset = dataset.filter(marketing_filter)
dataset = dataset.filter(tone_complexity_filter)
print("After all filters:", len(dataset))

# Cap at 2000 so training stays within Colab session limits
dataset = dataset.shuffle(seed=42).select(range(min(2000, len(dataset))))
print("Final working dataset size:", len(dataset))
print("Example:", dataset[0])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/668 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/16.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Raw dataset size: 50000
Columns: ['task', 'input_text', 'output_text', 'Tone', 'Target Audience', 'Verbosity', 'Style', 'Complexity Level', 'Emotion', 'Purpose', 'Vocabulary Range']


...

Filter:   0%|          | 0/5726 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5113 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2874 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2668 [00:00<?, ? examples/s]

After all filters: 369
Final working dataset size: 369
Example: {'task': 'paraphrasing', 'input_text': "The company 's recent decision to cut back with employee benefits is a short - sighted move that will ultimately harm morale and productivity . It 's crucial to recognize that investing in your workforce is not just an cost , but an essential investment in the future success of the business .", 'output_text': 'The recent decision by the company to reduce employee benefits reflects a myopic approach that is likely to undermine both morale and productivity. It is imperative to acknowledge that nurturing the workforce is not merely an expense, but a vital investment in the long-term prosperity of the organization.', 'Tone': 'Professional', 'Target Audience': 'Business professionals', 'Verbosity': 'Balanced', 'Style': 'Argumentative', 'Complexity Level': 'Intermediate', 'Emotion': 'Empathetic', 'Purpose': 'Entertain', 'Vocabulary Range': 'Creative'}


In [5]:
split1 = dataset.train_test_split(test_size=0.2, seed=42)
split2 = split1["test"].train_test_split(test_size=0.5, seed=42)

train_dataset = split1["train"]
val_dataset   = split2["train"]
test_dataset  = split2["test"]

print(f"Train : {len(train_dataset)}")
print(f"Val   : {len(val_dataset)}")
print(f"Test  : {len(test_dataset)}")

Train : 295
Val   : 37
Test  : 37


In [6]:
INSTRUCTION = (
    "Rewrite the following statement in a professional executive business tone."
)

def formatting_func(example):
    return (
        f"### Instruction:\n{INSTRUCTION}\n\n"
        f"### Input:\n{example['input_text']}\n\n"
        f"### Response:\n{example['output_text']}"
    )

def build_prompt_only(input_text):
    """Used at inference time — no response included."""
    return (
        f"### Instruction:\n{INSTRUCTION}\n\n"
        f"### Input:\n{input_text}\n\n"
        f"### Response:\n"
    )


In [7]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float32,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
tokenizer.model_max_length = 256   # ← add this

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float32,
)
model.config.use_cache = False  # disable KV-cache during training

print("Model loaded — parameters:", sum(p.numel() for p in model.parameters()) / 1e6, "M")


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded — parameters: 615.606272 M


In [8]:
def generate_response(model, tokenizer, input_text, max_new_tokens=80):
    """Generate a single response from the model."""
    prompt = build_prompt_only(input_text)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    # Decode only the newly generated tokens
    new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def evaluate_model(model, tokenizer, eval_dataset, n_samples=50, label="Model"):
    model.eval()
    sample = eval_dataset.select(range(min(n_samples, len(eval_dataset))))
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

    predictions, references = [], []
    r1_scores, r2_scores, rL_scores = [], [], []

    for ex in sample:
        pred = generate_response(model, tokenizer, ex["input_text"])
        ref  = ex["output_text"]
        predictions.append(pred)
        references.append(ref)

        scores = scorer.score(ref, pred)
        r1_scores.append(scores["rouge1"].fmeasure)
        r2_scores.append(scores["rouge2"].fmeasure)
        rL_scores.append(scores["rougeL"].fmeasure)

    # BLEU using nltk
    smooth  = SmoothingFunction().method1
    tokenised_preds = [p.split() for p in predictions]
    tokenised_refs  = [[r.split()] for r in references]
    bleu = corpus_bleu(tokenised_refs, tokenised_preds, smoothing_function=smooth) * 100

    results = {
        "ROUGE-1": round(np.mean(r1_scores), 4),
        "ROUGE-2": round(np.mean(r2_scores), 4),
        "ROUGE-L": round(np.mean(rL_scores), 4),
        "BLEU"   : round(bleu, 2),
    }

    print(f"\n{'='*50}\n  Evaluation — {label}\n{'='*50}")
    for k, v in results.items():
        print(f"  {k:<12}: {v}")

    print("\n  Qualitative sample (first 3):")
    for i in range(min(3, len(predictions))):
        print(f"\n  Input     : {sample[i]['input_text'][:120]}")
        print(f"  Reference : {references[i][:120]}")
        print(f"  Predicted : {predictions[i][:120]}")

    return results, predictions, references

In [9]:
print("Running BASELINE evaluation …")
baseline_results, baseline_preds, baseline_refs = evaluate_model(
    model, tokenizer, test_dataset, n_samples=50, label="Baseline (no fine-tuning)"
)


Running BASELINE evaluation …

  Evaluation — Baseline (no fine-tuning)
  ROUGE-1     : 0.3312
  ROUGE-2     : 0.0743
  ROUGE-L     : 0.2335
  BLEU        : 2.08

  Qualitative sample (first 3):

  Input     : an study of quantum mechanics has revolutionized our understanding of the fundamental nature of reality . Researchers ha
  Reference : The exploration of quantum mechanics has profoundly transformed our comprehension of the foundational aspects of existen
  Predicted : Researchers have made significant strides in developing theories and models that explain the behavior of particles at th

  Input     : The implementation of blockchain technology in supply chain management has revolutionize the industry , providing unprec
  Reference : The integration of blockchain technology within supply chain management has heralded a new era of transparency and opera
  Predicted : Rather than deter smaller enterprises from embracing blockchain technology in supply chain management, the stateme

In [10]:
lora_config = LoraConfig(
    r=16,                    # rank — higher = more capacity
    lora_alpha=32,           # scaling factor
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # all attention projections
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


In [12]:
OUTPUT_DIR = "./results"

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=20,
    eval_strategy="no",   # ← updated
    save_strategy="no",
    fp16=False,
    bf16=False,
    packing=False,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    # eval_dataset=val_dataset,   ← remove this line
    args=training_args,
    formatting_func=formatting_func,
    processing_class=tokenizer,
)

print("Starting QLoRA training …")
train_result = trainer.train()
print("\nTraining complete.")
print("Training metrics:", train_result.metrics)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Applying formatting function to train dataset:   0%|          | 0/295 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/295 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/295 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/295 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Starting QLoRA training …


Step,Training Loss
20,1.464653



Training complete.
Training metrics: {'train_runtime': 247.3526, 'train_samples_per_second': 2.385, 'train_steps_per_second': 0.154, 'total_flos': 670827415117824.0, 'train_loss': 1.2665784233494808}


In [13]:
SAVE_DIR = "./executive_rewriter_qlora"
trainer.model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Adapter saved to {SAVE_DIR}")

Adapter saved to ./executive_rewriter_qlora


In [14]:
print("Running FINE-TUNED model evaluation …")
finetuned_results, finetuned_preds, finetuned_refs = evaluate_model(
    model, tokenizer, test_dataset, n_samples=50, label="QLoRA Fine-Tuned"
)

Running FINE-TUNED model evaluation …

  Evaluation — QLoRA Fine-Tuned
  ROUGE-1     : 0.4851
  ROUGE-2     : 0.1969
  ROUGE-L     : 0.4034
  BLEU        : 12.02

  Qualitative sample (first 3):

  Input     : an study of quantum mechanics has revolutionized our understanding of the fundamental nature of reality . Researchers ha
  Reference : The exploration of quantum mechanics has profoundly transformed our comprehension of the foundational aspects of existen
  Predicted : The ongoing exploration of quantum mechanics has profoundly transformed our understanding of the fundamental nature of r

  Input     : The implementation of blockchain technology in supply chain management has revolutionize the industry , providing unprec
  Reference : The integration of blockchain technology within supply chain management has heralded a new era of transparency and opera
  Predicted : The integration of blockchain technology in the supply chain has fundamentally transformed the industry, offering 

In [15]:
print("\n" + "="*60)
print("  COMPARATIVE RESULTS SUMMARY")
print("="*60)

metrics = ["ROUGE-1", "ROUGE-2", "ROUGE-L", "BLEU"]
comparison_df = pd.DataFrame(
    {
        "Metric"   : metrics,
        "Baseline" : [baseline_results[m] for m in metrics],
        "QLoRA"    : [finetuned_results[m] for m in metrics],
        "Δ Change" : [
            round(finetuned_results[m] - baseline_results[m], 4)
            for m in metrics
        ],
    }
)
print(comparison_df.to_string(index=False))

print("\n── Qualitative Comparison (5 test examples) ──")
test_sample = test_dataset.select(range(min(5, len(test_dataset))))
for i, ex in enumerate(test_sample):
    base_pred = generate_response(model, tokenizer, ex["input_text"])  # adapter already active
    print(f"\nExample {i+1}")
    print(f"  Input     : {ex['input_text']}")
    print(f"  Reference : {ex['output_text']}")
    print(f"  Baseline  : {baseline_preds[i] if i < len(baseline_preds) else 'N/A'}")
    print(f"  QLoRA     : {base_pred}")



  COMPARATIVE RESULTS SUMMARY
 Metric  Baseline   QLoRA  Δ Change
ROUGE-1    0.3312  0.4851    0.1539
ROUGE-2    0.0743  0.1969    0.1226
ROUGE-L    0.2335  0.4034    0.1699
   BLEU    2.0800 12.0200    9.9400

── Qualitative Comparison (5 test examples) ──

Example 1
  Input     : an study of quantum mechanics has revolutionized our understanding of the fundamental nature of reality . Researchers have madeed significant strides in developing theories and models that explain the behavior of particles at the subatomic level . These advancements have not only deepeneds our knowledge but also opened up new avenues for technological innovation .
  Reference : The exploration of quantum mechanics has profoundly transformed our comprehension of the foundational aspects of existence. Scholars have achieved notable progress in constructing theories and frameworks that elucidate the conduct of particles within the realm of subatomic dimensions. These breakthroughs have not only enriched our co

In [16]:
demo_inputs = [
    "Revenue declined by 5% due to reduced enterprise client spending.",
    "We lost customers because the product was too expensive.",
    "The team missed the quarterly target because of supply chain issues.",
    "Our marketing spend went up but conversions went down this month.",
]

print("\n── Live Inference Demo ──")
for text in demo_inputs:
    response = generate_response(model, tokenizer, text)
    print(f"\nInput   : {text}")
    print(f"Output  : {response}")



── Live Inference Demo ──

Input   : Revenue declined by 5% due to reduced enterprise client spending.
Output  : The decline in revenue was attributed to a reduction in enterprise client spending.

Input   : We lost customers because the product was too expensive.
Output  : The pricing strategy implemented by the company has resulted in a significant decline in customer acquisition.

Input   : The team missed the quarterly target because of supply chain issues.
Output  : The team's performance was subpar due to supply chain disruptions.

Input   : Our marketing spend went up but conversions went down this month.
Output  : Our marketing expenditure increased, but the conversion rate was significantly lower in this month.
